In [13]:
#%pip install kaleido
#%pip install mplcursors
#%pip install matplotlib widget

In [14]:
# all libraries used in this notebook
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' #only critical errors
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    LayerNormalization, Input, Conv1D, MaxPooling1D, Flatten,
    Concatenate, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')



In [15]:
df = pd.read_csv("NVDA.csv")

In [16]:
df.head(5)

,Date,Adj Close,Close,High,Low,Open,Volume
0,1999-01-22,0.037615,0.041016,0.048828,0.038802,0.043750,2714688000
1,1999-01-25,0.041556,0.045313,0.045833,0.041016,0.044271,510480000
2,1999-01-26,0.038331,0.041797,0.046745,0.041146,0.045833,343200000
3,1999-01-27,0.038212,0.041667,0.042969,0.039583,0.041927,244368000
4,1999-01-28,0.038092,0.041536,0.041927,0.041276,0.041667,227520000


In [17]:
df = df.sort_values('Date').reset_index(drop=True)

print("Shape:", df.shape)
print(df.head())

Shape: (6558, 7)
         Date  Adj Close     Close      High       Low      Open      Volume
0  1999-01-22   0.037615  0.041016  0.048828  0.038802  0.043750  2714688000
1  1999-01-25   0.041556  0.045313  0.045833  0.041016  0.044271   510480000
2  1999-01-26   0.038331  0.041797  0.046745  0.041146  0.045833   343200000
3  1999-01-27   0.038212  0.041667  0.042969  0.039583  0.041927   244368000
4  1999-01-28   0.038092  0.041536  0.041927  0.041276  0.041667   227520000


### Null Analysis


In [18]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({'Nulls': nulls, 'Porcentage (%)': null_pct})
print(null_report)

           Nulls  Porcentage (%)
Date           0             0.0
Adj Close      0             0.0
Close          0             0.0
High           0             0.0
Low            0             0.0
Open           0             0.0
Volume         0             0.0


In [19]:
dupes = df.duplicated().sum()
print(f"\nDuplicates: {dupes}")
date_dupes = df['Date'].duplicated().sum()
print(f"Duplicated dates: {date_dupes}")


Duplicates: 0
Duplicated dates: 0


In [20]:
df.describe()

,Adj Close,Close,High,Low,Open,Volume
count,6558.000000,6558.000000,6558.000000,6558.000000,6558.000000,6.558000e+03
mean,8.768532,8.795447,8.956567,8.618315,8.795850,5.991103e+08
std,23.907205,23.904882,24.349618,23.419200,23.922708,4.307236e+08
min,0.031286,0.034115,0.035547,0.033333,0.034896,1.968000e+07
25%,0.257739,0.281042,0.288511,0.273354,0.280810,3.384780e+08
50%,0.437176,0.466083,0.472875,0.459250,0.466584,5.002635e+08
75%,4.597059,4.644625,4.724000,4.588750,4.632437,7.307002e+08
max,149.429993,149.429993,153.130005,147.820007,153.029999,9.230856e+09


In [21]:
print(f"\nStart date: {df['Date'].min()}")
print(f"End date: {df['Date'].max()}")
print(f"Total trading days: {len(df)}")


Start date: 1999-01-22
End date: 2025-02-14
Total trading days: 6558


In [22]:
date_range = pd.bdate_range(start=df['Date'].min(), end=df['Date'].max())
missing_dates = date_range.difference(df['Date'])
print(f"Missing business days (possible gaps): {len(missing_dates)}")

Missing business days (possible gaps): 6801


### OUTLIERS


In [23]:

print("\n── Outliers by column (IQR method) ──")
num_cols = ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"  {col}: {len(outliers)} outliers")



── Outliers by column (IQR method) ──
  Adj Close: 1139 outliers
  Close: 1138 outliers
  High: 1136 outliers
  Low: 1134 outliers
  Open: 1136 outliers
  Volume: 336 outliers


In [ ]:
#Reduce data set to 2020-2024 for better model performance and relevance
df = df[df['Date'] >= '2020-01-01'].reset_index(drop=True)
df['Date'] = pd.to_datetime(df['Date'])
print(f"Data: {len(df)} | {df['Date'].min().date()} → {df['Date'].max().date()}")

AttributeError: 'str' object has no attribute 'date'

### Features for model

In [ ]:

def add_features(df):
    d = df.copy()

    # Returns
    d['ret_1']  = d['Adj Close'].pct_change(1)
    d['ret_3']  = d['Adj Close'].pct_change(3)
    d['ret_5']  = d['Adj Close'].pct_change(5)
    d['ret_10'] = d['Adj Close'].pct_change(10)
    d['ret_20'] = d['Adj Close'].pct_change(20)

    # MAs relative to price
    for w in [5, 10, 20, 50]:
        d[f'ma{w}_ratio'] = d['Adj Close'].rolling(w).mean() / d['Adj Close'] - 1

    # Volatibility (std de retornos)
    d['vol_5']  = d['ret_1'].rolling(5).std()
    d['vol_10'] = d['ret_1'].rolling(10).std()
    d['vol_20'] = d['ret_1'].rolling(20).std()

    # RSI
    delta = d['Adj Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-8)))
    d['rsi'] = d['rsi'] / 100  # normalizar 0-1

    # MACD
    ema12 = d['Adj Close'].ewm(span=12).mean()
    ema26 = d['Adj Close'].ewm(span=26).mean()
    macd  = ema12 - ema26
    d['macd_norm']   = macd / d['Adj Close']
    d['macd_signal'] = macd.ewm(span=9).mean() / d['Adj Close']

    # Bollinger
    bb_mid = d['Adj Close'].rolling(20).mean()
    bb_std = d['Adj Close'].rolling(20).std()
    d['bb_pos'] = (d['Adj Close'] - bb_mid) / (2 * bb_std + 1e-8)  # -1 a +1

    # Candle
    d['hl_pct']   = (d['High'] - d['Low']) / d['Adj Close']
    d['oc_pct']   = (d['Adj Close'] - d['Open']) / d['Open']

    # Volume
    d['vol_ratio'] = np.log1p(d['Volume']) / np.log1p(d['Volume'].rolling(20).mean() + 1e-8) - 1

    return d.dropna().reset_index(drop=True)

df = add_features(df)
print(f"Post feature engineering: {len(df)} data")

### Normalization

In [ ]:
FEATURE_COLS = [
    'ret_1', 'ret_3', 'ret_5', 'ret_10', 'ret_20',
    'ma5_ratio', 'ma10_ratio', 'ma20_ratio', 'ma50_ratio',
    'vol_5', 'vol_10', 'vol_20',
    'rsi', 'macd_norm', 'macd_signal', 'bb_pos',
    'hl_pct', 'oc_pct', 'vol_ratio'
]
N_FEATURES  = len(FEATURE_COLS)
WINDOW_SIZE = 30

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
features_scaled = scaler.fit_transform(df[FEATURE_COLS])

prices = df['Adj Close'].values

def create_sequences(features, prices, window):
    """
    X: features escaladas
    y: retorno acumulado normalizado al inicio de la ventana
       (precio_siguiente / precio_inicio_ventana - 1)
    """
    Xs, ys, price_refs = [], [], []
    for i in range(window, len(features)):
        Xs.append(features[i-window:i])
        # Target: cuánto sube/baja el precio del día siguiente
        # relativo al precio al inicio de la ventana
        ref_price = prices[i - window]
        ys.append((prices[i] / ref_price) - 1)
        price_refs.append(ref_price)
    return np.array(Xs), np.array(ys), np.array(price_refs)

X_all, y_all, refs_all = create_sequences(features_scaled, prices, WINDOW_SIZE)
print(f"Sequences: X={X_all.shape}, y={y_all.shape}")

### Division Dataset

In [ ]:
n         = len(X_all)
train_end = int(n * 0.72)
val_end   = int(n * 0.86)

X_train, y_train = X_all[:train_end],        y_all[:train_end]
X_val,   y_val   = X_all[train_end:val_end],  y_all[train_end:val_end]
X_test,  y_test  = X_all[val_end:],           y_all[val_end:]
refs_test        = refs_all[val_end:]

print(f"Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

### Model

In [ ]:
def build_model(window, n_features):
    inputs = Input(shape=(window, n_features))

    # CNN branch — short-term patterns
    c = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    c = Conv1D(32, kernel_size=3, activation='relu', padding='same')(c)
    c = GlobalAveragePooling1D()(c)

    # LSTM branch — long-term dependencies
    x = Bidirectional(LSTM(96, return_sequences=True))(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)
    x = Bidirectional(LSTM(48, return_sequences=False))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.20)(x)

    # Fusion
    merged = Concatenate()([x, c])
    out = Dense(64, activation='relu')(merged)
    out = Dropout(0.15)(out)
    out = Dense(32, activation='relu')(out)
    out = Dense(1)(out)

    return Model(inputs, out)

model = build_model(WINDOW_SIZE, N_FEATURES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss='huber',
    metrics=['mae']
)
model.summary()


### Training

In [ ]:

callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True,
                  monitor='val_loss', verbose=1),
    ReduceLROnPlateau(factor=0.4, patience=8,
                      min_lr=5e-7, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


### Evaluation

In [ ]:

y_pred_norm = model.predict(X_test, verbose=0).flatten()
y_real_norm = y_test.flatten()

prices_pred = refs_test * (1 + y_pred_norm)
prices_real = refs_test * (1 + y_real_norm)

# Compparison against real prices from the dataframe (not just reconstructed from refs)
test_start_idx  = val_end + WINDOW_SIZE
prices_real_df  = df['Adj Close'].iloc[test_start_idx:test_start_idx + len(y_pred_norm)].values

mae  = mean_absolute_error(prices_real_df, prices_pred)
rmse = np.sqrt(mean_squared_error(prices_real_df, prices_pred))
r2   = r2_score(prices_real_df, prices_pred)
mape = np.mean(np.abs((prices_real_df - prices_pred) / (prices_real_df + 1e-8))) * 100

# Directional accuracy: % de veces que el modelo predice la dirección correcta del movimiento
dir_acc = np.mean(np.sign(y_pred_norm) == np.sign(y_real_norm)) * 100

print("\n" + "="*60)
print("  MÉTRICAS — TEST SET")
print("="*60)
print(f"  MAE  : ${mae:.4f}")
print(f"  RMSE : ${rmse:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
print(f"  Directional accuracy: {dir_acc:.1f}%")

test_dates = df['Date'].iloc[test_start_idx:test_start_idx + len(y_pred_norm)].values

### Forcasting 30 days

In [ ]:

last_features = features_scaled[-WINDOW_SIZE:]  # última ventana real
forecast_prices = []
forecast_dates  = pd.bdate_range(
    start=df['Date'].max() + pd.Timedelta(days=1), periods=30
)

current_window = last_features.copy()
last_price     = df['Adj Close'].iloc[-1]

for step in range(30):
    ref_price = prices[-(WINDOW_SIZE - step)] if step < WINDOW_SIZE else last_price

    pred_norm = model.predict(
        current_window[np.newaxis, :, :], verbose=0
    )[0, 0]

    new_price = ref_price * (1 + pred_norm)
    forecast_prices.append(new_price)

    # Construir nueva fila de features (aproximada)
    new_ret    = pred_norm  # retorno relativo al inicio ventana (aprox)
    new_row    = current_window[-1].copy()
    new_row[0] = np.clip(new_ret, -0.5, 0.5)  # ret_1 aprox
    current_window = np.vstack([current_window[1:], new_row])

forecast_prices = np.array(forecast_prices)

In [ ]:


# ── 9. VISUALIZACIÓN ─────────────────────────────────────────
fig = plt.figure(figsize=(18, 16), facecolor='#0f0f1a')
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

C_REAL = '#00e5ff'
C_PRED = '#ff6b35'
C_FORE = '#a8ff78'
C_LOSS = '#ff4d6d'
C_VAL  = '#ffd60a'
BG     = '#1a1a2e'

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white')
    ax.grid(alpha=0.15, color='white')
    for s in ax.spines.values():
        s.set_color('#444')

# Plot 1: Serie histórica
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df['Date'], df['Adj Close'], color=C_REAL, linewidth=1.2, label='Precio real')
ax1.axvline(df['Date'].iloc[train_end + WINDOW_SIZE], color='white', linestyle='--', alpha=0.4)
ax1.axvline(df['Date'].iloc[val_end + WINDOW_SIZE],   color=C_VAL,  linestyle='--', alpha=0.5)
ymax = df['Adj Close'].max()
ax1.text(df['Date'].iloc[train_end + WINDOW_SIZE], ymax*0.85, ' Train|Val', color='white', fontsize=8)
ax1.text(df['Date'].iloc[val_end + WINDOW_SIZE],   ymax*0.85, ' Val|Test',  color=C_VAL,   fontsize=8)
style_ax(ax1, 'NVDA — Serie histórica (2020 → actualidad)')
ax1.set_ylabel('Precio (USD)', color='white')
ax1.legend(facecolor=BG, labelcolor='white', fontsize=9)

# Plot 2: Predicción vs Real
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(test_dates, prices_real_df, color=C_REAL, linewidth=1.5, label='Real',       alpha=0.9)
ax2.plot(test_dates, prices_pred,    color=C_PRED, linewidth=1.5, label='Predicción', alpha=0.9, linestyle='--')
ax2.fill_between(test_dates, prices_real_df, prices_pred, alpha=0.08, color=C_PRED)
style_ax(ax2, f'Test — Real vs Predicción  |  MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.3f}  Dir={dir_acc:.1f}%')
ax2.set_ylabel('Precio (USD)', color='white')
ax2.legend(facecolor=BG, labelcolor='white', fontsize=9)

# Plot 3: Forecast
ax3 = fig.add_subplot(gs[2, 0])
recent = df.tail(90)
ax3.plot(recent['Date'], recent['Adj Close'], color=C_REAL, linewidth=1.5, label='Histórico')
ax3.plot(forecast_dates, forecast_prices, color=C_FORE, linewidth=2,
         marker='o', markersize=3, label='Forecast 30d')
ax3.axvline(df['Date'].max(), color='white', linestyle='--', alpha=0.4)
style_ax(ax3, 'Forecast — Próximos 30 días hábiles')
ax3.set_ylabel('Precio (USD)', color='white')
ax3.tick_params(labelrotation=30)
ax3.legend(facecolor=BG, labelcolor='white', fontsize=8)

# Plot 4: Loss
ax4 = fig.add_subplot(gs[2, 1])
ax4.plot(history.history['loss'],     color=C_REAL, linewidth=1.5, label='Train')
ax4.plot(history.history['val_loss'], color=C_LOSS, linewidth=1.5, label='Val', linestyle='--')
style_ax(ax4, 'Curva de Aprendizaje')
ax4.set_xlabel('Época', color='white')
ax4.set_ylabel('Huber Loss', color='white')
ax4.legend(facecolor=BG, labelcolor='white', fontsize=9)

plt.suptitle('NVDA Stock — CNN + Bidirectional LSTM (v3)', color='white',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('nvda_lstm_v3_results.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
print("\n✓ Gráfica guardada: nvda_lstm_v3_results.png")

# ── 10. TABLA FORECAST ────────────────────────────────────────
print("\n── Forecast próximos 30 días ──")
for d, p in zip(forecast_dates[:10], forecast_prices[:10]):
    delta_pct = (p - df['Adj Close'].iloc[-1]) / df['Adj Close'].iloc[-1] * 100
    sign = "▲" if p > df['Adj Close'].iloc[-1] else "▼"
    print(f"  {d.date()}  →  ${p:.4f}  {sign} {delta_pct:+.2f}% vs hoy")
print("  ...")
print(f"  {forecast_dates[-1].date()}  →  ${forecast_prices[-1]:.4f}")
delta = forecast_prices[-1] - df['Adj Close'].iloc[-1]
print(f"\nPrecio actual:   ${df['Adj Close'].iloc[-1]:.4f}")
print(f"Precio day+30:   ${forecast_prices[-1]:.4f}")
print(f"Cambio estimado: {'+' if delta >= 0 else ''}{delta:.4f} USD ({delta/df['Adj Close'].iloc[-1]*100:+.2f}%)")
print(f"Dirección correcta en test: {dir_acc:.1f}%")

In [ ]:
#%pip install plotly


In [ ]:
#%pip install ipympl 

In [ ]:
# ── CELDA 2: visualización completa ──
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mplcursors


fig = plt.figure(figsize=(18, 16), facecolor='#0f0f1a')
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white')
    ax.grid(alpha=0.15, color='white')
    for s in ax.spines.values():
        s.set_color('#444')

# ── Plot 1: Serie histórica ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
l1, = ax1.plot(df['Date'], df['Adj Close'], color=C_REAL, linewidth=1.2, label='Precio real')
ax1.axvline(df['Date'].iloc[train_end + WINDOW_SIZE], color='white', linestyle='--', alpha=0.4)
ax1.axvline(df['Date'].iloc[val_end   + WINDOW_SIZE], color=C_VAL,   linestyle='--', alpha=0.5)
ymax = df['Adj Close'].max()
ax1.text(df['Date'].iloc[train_end + WINDOW_SIZE], ymax*0.85, ' Train|Val', color='white', fontsize=8)
ax1.text(df['Date'].iloc[val_end   + WINDOW_SIZE], ymax*0.85, ' Val|Test',  color=C_VAL,   fontsize=8)
style_ax(ax1, 'NVDA — Serie histórica (2020 → actualidad)')
ax1.set_ylabel('Precio (USD)', color='white')
ax1.legend(facecolor=BG, labelcolor='white', fontsize=9)

# ── Plot 2: Real vs Predicción ────────────────────────────────
ax2 = fig.add_subplot(gs[1, :])
l2, = ax2.plot(test_dates, prices_real_df, color=C_REAL, linewidth=1.5, label='Real')
l3, = ax2.plot(test_dates, prices_pred,    color=C_PRED, linewidth=1.5, label='Predicción', linestyle='--')
ax2.fill_between(test_dates, prices_real_df, prices_pred, alpha=0.08, color=C_PRED)
style_ax(ax2, f'Test — Real vs Predicción  |  MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.3f}  Dir={dir_acc:.1f}%')
ax2.set_ylabel('Precio (USD)', color='white')
ax2.legend(facecolor=BG, labelcolor='white', fontsize=9)

# ── Plot 3: Forecast ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, 0])
recent    = df.tail(90)
last_price = df['Adj Close'].iloc[-1]
l4, = ax3.plot(recent['Date'], recent['Adj Close'], color=C_REAL, linewidth=1.5, label='Histórico')
l5, = ax3.plot(forecast_dates, forecast_prices,     color=C_FORE, linewidth=2,
               marker='o', markersize=3, label='Forecast 30d')
ax3.axvline(df['Date'].max(), color='white', linestyle='--', alpha=0.4)
style_ax(ax3, 'Forecast — Próximos 30 días hábiles')
ax3.set_ylabel('Precio (USD)', color='white')
ax3.tick_params(labelrotation=30)
ax3.legend(facecolor=BG, labelcolor='white', fontsize=8)

# ── Plot 4: Loss ──────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 1])
l6, = ax4.plot(history.history['loss'],     color=C_REAL, linewidth=1.5, label='Train')
l7, = ax4.plot(history.history['val_loss'], color=C_LOSS, linewidth=1.5, label='Val', linestyle='--')
style_ax(ax4, 'Curva de Aprendizaje')
ax4.set_xlabel('Época', color='white')
ax4.set_ylabel('Huber Loss', color='white')
ax4.legend(facecolor=BG, labelcolor='white', fontsize=9)

# ── Hover ─────────────────────────────────────────────────────
def style_hover(sel):
    sel.annotation.get_bbox_patch().set(fc="#1a1a2e", alpha=0.9, ec="#444")
    sel.annotation.set_color("white")
    sel.annotation.set_fontsize(9)

for line, fmt in [
    (l1, lambda sel: f"📅 {df['Date'].iloc[sel.index].date()}\n💲 ${sel.target[1]:.2f}"),
    (l2, lambda sel: f"📅 {test_dates[sel.index].date()}\n💲 Real: ${sel.target[1]:.4f}"),
    (l3, lambda sel: f"📅 {test_dates[sel.index].date()}\n🔮 Pred: ${sel.target[1]:.4f}"),
    (l5, lambda sel: f"📅 {forecast_dates[sel.index].date()}\n🔮 ${sel.target[1]:.4f}  ({(sel.target[1]-last_price)/last_price*100:+.2f}%)"),
    (l6, lambda sel: f"Época {sel.index+1}\nTrain: {sel.target[1]:.6f}"),
    (l7, lambda sel: f"Época {sel.index+1}\nVal:   {sel.target[1]:.6f}"),
]:
    c = mplcursors.cursor(line, hover=True)
    c.connect("add", lambda sel, f=fmt: (sel.annotation.set_text(f(sel)), style_hover(sel)))

plt.suptitle('NVDA Stock — CNN + Bidirectional LSTM (v3)', color='white',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mplcursors

fig = plt.figure(figsize=(18, 16), facecolor='#0f0f1a')
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

C_REAL = '#00e5ff'
C_PRED = '#ff6b35'
C_FORE = '#a8ff78'
C_LOSS = '#ff4d6d'
C_VAL  = '#ffd60a'
BG     = '#1a1a2e'

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='white')
    ax.grid(alpha=0.15, color='white')
    for s in ax.spines.values():
        s.set_color('#444')

# Plot 1: Serie histórica
ax1 = fig.add_subplot(gs[0, :])
l1, = ax1.plot(df['Date'], df['Adj Close'], color=C_REAL, linewidth=1.2, label='Precio real')
ax1.axvline(df['Date'].iloc[train_end + WINDOW_SIZE], color='white', linestyle='--', alpha=0.4)
ax1.axvline(df['Date'].iloc[val_end + WINDOW_SIZE],   color=C_VAL,  linestyle='--', alpha=0.5)
ymax = df['Adj Close'].max()
ax1.text(df['Date'].iloc[train_end + WINDOW_SIZE], ymax*0.85, ' Train|Val', color='white', fontsize=8)
ax1.text(df['Date'].iloc[val_end + WINDOW_SIZE],   ymax*0.85, ' Val|Test',  color=C_VAL,   fontsize=8)
style_ax(ax1, 'NVDA — Serie histórica (2020 → actualidad)')
ax1.set_ylabel('Precio (USD)', color='white')
ax1.legend(facecolor=BG, labelcolor='white', fontsize=9)

# Plot 2: Real vs Predicción
ax2 = fig.add_subplot(gs[1, :])
l2, = ax2.plot(test_dates, prices_real_df, color=C_REAL, linewidth=1.5, label='Real')
l3, = ax2.plot(test_dates, prices_pred,    color=C_PRED, linewidth=1.5, label='Predicción', linestyle='--')
ax2.fill_between(test_dates, prices_real_df, prices_pred, alpha=0.08, color=C_PRED)
style_ax(ax2, f'Test — Real vs Predicción  |  MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.3f}  Dir={dir_acc:.1f}%')
ax2.set_ylabel('Precio (USD)', color='white')
ax2.legend(facecolor=BG, labelcolor='white', fontsize=9)

# Plot 3: Forecast
ax3 = fig.add_subplot(gs[2, 0])
recent = df.tail(90)
l4, = ax3.plot(recent['Date'], recent['Adj Close'], color=C_REAL, linewidth=1.5, label='Histórico')
last_price = df['Adj Close'].iloc[-1]
l5, = ax3.plot(forecast_dates, forecast_prices, color=C_FORE, linewidth=2,
               marker='o', markersize=3, label='Forecast 30d')
ax3.axvline(df['Date'].max(), color='white', linestyle='--', alpha=0.4)
style_ax(ax3, 'Forecast — Próximos 30 días hábiles')
ax3.set_ylabel('Precio (USD)', color='white')
ax3.tick_params(labelrotation=30)
ax3.legend(facecolor=BG, labelcolor='white', fontsize=8)

# Plot 4: Loss
ax4 = fig.add_subplot(gs[2, 1])
l6, = ax4.plot(history.history['loss'],     color=C_REAL, linewidth=1.5, label='Train')
l7, = ax4.plot(history.history['val_loss'], color=C_LOSS, linewidth=1.5, label='Val', linestyle='--')
style_ax(ax4, 'Curva de Aprendizaje')
ax4.set_xlabel('Época', color='white')
ax4.set_ylabel('Huber Loss', color='white')
ax4.legend(facecolor=BG, labelcolor='white', fontsize=9)

# ── Hover con mplcursors ──────────────────────────────────────
for line, fmt in [
    (l1, lambda sel: f" {df['Date'].iloc[sel.index].date()}\n💲 ${sel.target[1]:.2f}"),
    (l2, lambda sel: f" {test_dates[sel.index].date()}\n💲 Real: ${sel.target[1]:.4f}"),
    (l3, lambda sel: f" {test_dates[sel.index].date()}\n🔮 Pred: ${sel.target[1]:.4f}"),
    (l5, lambda sel: f" {forecast_dates[sel.index].date()}\n🔮 ${sel.target[1]:.4f}  ({(sel.target[1]-last_price)/last_price*100:+.2f}%)"),
    (l6, lambda sel: f"Época {sel.index+1}\nTrain Loss: {sel.target[1]:.6f}"),
    (l7, lambda sel: f"Época {sel.index+1}\nVal Loss:   {sel.target[1]:.6f}"),
]:
    cursor = mplcursors.cursor(line, hover=True)
    cursor.connect("add", lambda sel, f=fmt: sel.annotation.set_text(f(sel)))

plt.suptitle('NVDA Stock — CNN + Bidirectional LSTM (v3)', color='white',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()